In [1]:
import getml
import mlflow
import getml_mlflow

In [2]:
mlflow.set_tracking_uri("http://localhost:5000")
getml_mlflow.autolog()

In [3]:
getml.set_project("interstate94")

Output()

Connected to project 'interstate94'.

In [4]:
traffic = getml.datasets.load_interstate94(roles=False, units=False)

In [5]:
traffic.set_role("ds", getml.data.roles.time_stamp)
traffic.set_role("holiday", getml.data.roles.categorical)
traffic.set_role("traffic_volume", getml.data.roles.target)

In [6]:
split = getml.data.split.time(traffic, "ds", test=getml.data.time.datetime(2018, 3, 15))

In [7]:
time_series = getml.data.TimeSeries(
    population=traffic,
    split=split,
    time_stamps="ds",
    horizon=getml.data.time.hours(1),
    memory=getml.data.time.days(7),
    lagged_targets=True,
)

pipe = getml.pipeline.Pipeline(
    tags=["memory: 7d", "horizon: 1h", "fast_prop"],
    data_model=time_series.data_model,
    preprocessors=[getml.preprocessors.Seasonal()],
    feature_learners=[
        getml.feature_learning.FastProp(
            loss_function=getml.feature_learning.loss_functions.SquareLoss,
            num_threads=1,
            num_features=20,
        )
    ],
    predictors=[getml.predictors.XGBoostRegressor()],
)
pipe

Pipeline(data_model='population',
         feature_learners=['FastProp'],
         feature_selectors=[],
         include_categorical=False,
         loss_function='SquareLoss',
         peripheral=['traffic'],
         predictors=['XGBoostRegressor'],
         preprocessors=['Seasonal'],
         share_selected_features=0.5,
         tags=['memory: 7d', 'horizon: 1h', 'fast_prop'])

In [8]:
fit1 = pipe.fit(time_series.train)
print(fit1.id, pipe.id)

Experiment: <Experiment: artifact_location='mlflow-artifacts:/844494434965818253', creation_time=1734105340779, experiment_id='844494434965818253', last_update_time=1734105340779, lifecycle_stage='active', name='interstate94', tags={}>
Via Project Experiment ID: 844494434965818253
Run Info: <RunInfo: artifact_uri='mlflow-artifacts:/844494434965818253/1f86da621fdb4ad1a509821a2275bfb8/artifacts', end_time=None, experiment_id='844494434965818253', lifecycle_stage='active', run_id='1f86da621fdb4ad1a509821a2275bfb8', run_name='Pipeline-NOT_FITTED', run_uuid='1f86da621fdb4ad1a509821a2275bfb8', start_time=1738076444530, status='RUNNING', user_id='unknown'>
Via Pipeline Run Experiment ID: 844494434965818253
GetMLDataset._resolve_source
GetMLDataset._compute_digest
GetMLDatasetSource.to_dict
GetMLDatasetSource._get_source_type
GetMLDataset.schema
GetMLDataset.schema
GetMLDataset.profile
GetMLDataset._resolve_source
GetMLDataset._compute_digest
GetMLDatasetSource.to_dict
GetMLDatasetSource._get_

Checking data model...

Output()

OK.

Output()

Trained pipeline.

2025/01/28 16:00:54 INFO mlflow.tracking._tracking_service.client: 🏃 View run fit at: http://localhost:5000/#/experiments/844494434965818253/runs/20b51157ee9145ab8d6f2e5d90bd1f8d.
2025/01/28 16:00:54 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/844494434965818253.
2025/01/28 16:00:54 INFO mlflow.tracking._tracking_service.client: 🏃 View run Pipeline-k2jZPu at: http://localhost:5000/#/experiments/844494434965818253/runs/1f86da621fdb4ad1a509821a2275bfb8.
2025/01/28 16:00:54 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/844494434965818253.


Time taken: 0:00:09.370907.

k2jZPu k2jZPu


In [9]:
fit2 = pipe.fit(time_series.train)
print(fit2.id, fit1.id, pipe.id)

Run Info: <RunInfo: artifact_uri='mlflow-artifacts:/844494434965818253/1f86da621fdb4ad1a509821a2275bfb8/artifacts', end_time=None, experiment_id='844494434965818253', lifecycle_stage='active', run_id='1f86da621fdb4ad1a509821a2275bfb8', run_name='Pipeline-NOT_FITTED', run_uuid='1f86da621fdb4ad1a509821a2275bfb8', start_time=1738076444530, status='RUNNING', user_id='unknown'>
Via Pipeline Run Experiment ID: 844494434965818253
Run Info: <RunInfo: artifact_uri='mlflow-artifacts:/844494434965818253/6e31c0cb62964dbc882a335a38981f98/artifacts', end_time=None, experiment_id='844494434965818253', lifecycle_stage='active', run_id='6e31c0cb62964dbc882a335a38981f98', run_name='Pipeline-k2jZPu', run_uuid='6e31c0cb62964dbc882a335a38981f98', start_time=1738076454449, status='RUNNING', user_id='unknown'>
Via Pipeline Run Experiment ID: 844494434965818253
GetMLDataset._resolve_source
GetMLDataset._compute_digest
GetMLDatasetSource.to_dict
GetMLDatasetSource._get_source_type
GetMLDataset.schema
GetMLData

Checking data model...

Output()

OK.

Output()

Trained pipeline.

2025/01/28 16:00:55 INFO mlflow.tracking._tracking_service.client: 🏃 View run fit at: http://localhost:5000/#/experiments/844494434965818253/runs/b788d01f93664c298bb79a229ae25b63.
2025/01/28 16:00:55 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/844494434965818253.
2025/01/28 16:00:55 INFO mlflow.tracking._tracking_service.client: 🏃 View run Pipeline-1rtxWB at: http://localhost:5000/#/experiments/844494434965818253/runs/6e31c0cb62964dbc882a335a38981f98.
2025/01/28 16:00:55 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/844494434965818253.


Time taken: 0:00:00.294766.

1rtxWB 1rtxWB 1rtxWB


In [10]:
pipe.score(time_series.test)

Output()

Run Info: <RunInfo: artifact_uri='mlflow-artifacts:/844494434965818253/6e31c0cb62964dbc882a335a38981f98/artifacts', end_time=None, experiment_id='844494434965818253', lifecycle_stage='active', run_id='6e31c0cb62964dbc882a335a38981f98', run_name='Pipeline-k2jZPu', run_uuid='6e31c0cb62964dbc882a335a38981f98', start_time=1738076454449, status='RUNNING', user_id='unknown'>
Via Pipeline Run Experiment ID: 844494434965818253
GetMLDataset._resolve_source
GetMLDataset._compute_digest
GetMLDatasetSource.to_dict
GetMLDatasetSource._get_source_type
GetMLDataset.schema
GetMLDataset.schema
GetMLDataset.profile
GetMLDataset._resolve_source
GetMLDataset._compute_digest
GetMLDatasetSource.to_dict
GetMLDatasetSource._get_source_type
GetMLDataset.schema
GetMLDataset.schema
GetMLDataset.profile


2025/01/28 16:00:55 INFO mlflow.tracking._tracking_service.client: 🏃 View run score at: http://localhost:5000/#/experiments/844494434965818253/runs/4f0df3988253469db1f2fcbad51d3e96.
2025/01/28 16:00:55 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/844494434965818253.


,date time,set used,target,mae,rmse,rsquared
0,2025-01-28 16:00:55,train,traffic_volume,200.4302,299.2045,0.9768
1,2025-01-28 16:00:55,test,traffic_volume,179.9515,269.631,0.9816


In [11]:
pipe.id

'1rtxWB'

In [12]:
# mlflow.data.dataset_source_registry.resolve_dataset_source("traffic.train.parquet")

In [13]:
ds = mlflow.data.dataset_source_registry.resolve_dataset_source("traffic.train.parquet")
print(ds)
print(ds.to_dict())
print(mlflow.get_artifact_uri("traffic.train.parquet"))

{'uri': 'traffic.train.parquet'}
mlflow-artifacts:/0/75e94654de194739aa2df59928aa4168/artifacts/traffic.train.parquet


/home/manuel/Projects/github/getml-mlflow/.venv/lib/python3.11/site-packages/mlflow/data/dataset_source_registry.py:149: UserWarning: Failed to determine whether UCVolumeDatasetSource can resolve source information for 'traffic.train.parquet'. Exception: 
  return _dataset_source_registry.resolve(
/home/manuel/Projects/github/getml-mlflow/.venv/lib/python3.11/site-packages/mlflow/data/dataset_source_registry.py:149: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


In [14]:
# run = mlflow.last_active_run()
# run.info.run_name = "ÄÄÄ"

In [15]:
pipe._mlflow_run_info

<RunInfo: artifact_uri='mlflow-artifacts:/844494434965818253/6e31c0cb62964dbc882a335a38981f98/artifacts', end_time=None, experiment_id='844494434965818253', lifecycle_stage='active', run_id='6e31c0cb62964dbc882a335a38981f98', run_name='Pipeline-k2jZPu', run_uuid='6e31c0cb62964dbc882a335a38981f98', start_time=1738076454449, status='RUNNING', user_id='unknown'>